# C5 lab: wire the answer step, then check its citations

Lesson: [`concepts/C5_generation_citations.md`](../concepts/C5_generation_citations.md), Part H.1. Concept card: [02 C5](../02_RAG_MASTER_LEARNING_PLAN.md#c5). Prerequisite: the [P7 lab](P7_grounded_prompt_lab.ipynb) (you built `build_prompt` and a first citation check there).

**How this lab works (blank-file first).**
- Each task has a **task cell**, an **empty code cell** for you, and a **CHECK cell** (`# CHECK`, do not edit). Run the CHECK cell after your code. If it fails, read the message and fix your code.
- This notebook contains **no solution code**. Write every line yourself. The reference script `sources/verification/C5/c5_measure.py` is a solution: open it only after an honest attempt plus a hint from your tutor.
- **Predict first.** Some tasks ask for a prediction before you compute. Fill it in honestly; it is part of the evidence.
- Tasks 1–8 make **no API calls**. The Groq call is tested with a **fake client** (`FakeGroq`), so you can test the wiring without a key. Task 9 is optional and calls Groq; it is switched off by default.
- Every answer string in the CHECK cells is a **hand-written toy answer**, not a model output.

## Setup (given)
Run this cell. It gives you:
- `n_tok` (the `o200k_harmony` token counter, as in P1/P7) and `_fp` (a fingerprint, so the CHECK cells can hide expected answers);
- **C4's retrieval result**, the input to C5. Each record is a dict: `rank`, `chunk_id`, `doc_id`, `page`, `score`, `text`. `RETRIEVED4` holds the **4 fixed chunks** retrieved for question D-002. They are v0 chunks (per-page recursive 1,000/200, course chunk ids `climate:ea07572562ac:1ca9c5a3b548:<seq>`) retrieved by a local MiniLM run (2026-10-08); this is C4's D-002 result. A demo fixture: **not** the v0 Cohere embedder and not an evaluation result;
- `ANSWER_V1_TXT`, the text of the template file `prompts/answer_v1.txt`, with two helpers: `split_template` and `split_sentences`;
- `FakeGroq`, a stand-in for `groq.Groq` that records the call and returns a canned reply (no network).

In [ ]:
import hashlib, json, re, time
from types import SimpleNamespace
import pymupdf, tiktoken

PDF = r"C:\Yash\New-Work\Book RAG\upstream\RAG_Techniques\data\Understanding_Climate_Change.pdf"
ENC = tiktoken.get_encoding("o200k_harmony")

def n_tok(text):
    """Number of o200k_harmony tokens in a string."""
    return len(ENC.encode(text, disallowed_special=()))

def _fp(x):
    """Fingerprint of a value. The CHECK cells compare fingerprints, not visible answers."""
    return hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()[:16]

def sha256_hex(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

pages = [p.get_text() for p in pymupdf.open(PDF)]

# ---- C4's retrieval result: (seq, page, doc_start, doc_end, score) for ranks 1-4 of each draft seed question (06 §6.7).
# v0 chunks (per-page recursive 1000/200), local MiniLM fallback, exact cosine; sources/verification/C5 S3 (= C4 S3).
# Offsets are DOCUMENT offsets into DOC_TEXT (pages joined with a newline). Demo fixture only.
DOC_TEXT = "\n".join(pages)
SNAPSHOT_ID, PARAMS_HASH = "ea07572562ac", "1ca9c5a3b548"
TOP4 = {
    "D-001": [(1, 1, 767, 1763, 0.4929), (0, 1, 0, 949, 0.4706), (7, 3, 5582, 6541, 0.4692), (8, 3, 6341, 7019, 0.4608)],
    "D-002": [(69, 24, 51414, 52396, 0.4606), (68, 23, 50756, 51413, 0.4513), (27, 10, 20555, 21506, 0.4386), (17, 6, 13112, 13935, 0.4385)],
    "D-003": [(68, 23, 50756, 51413, 0.6497), (27, 10, 20555, 21506, 0.632), (17, 6, 13112, 13935, 0.5351), (69, 24, 51414, 52396, 0.4791)],
    "D-004": [(67, 23, 49953, 50926, 0.421), (23, 8, 17650, 18299, 0.4035), (68, 23, 50756, 51413, 0.3604), (27, 10, 20555, 21506, 0.2136)],
    "D-005": [(3, 2, 2425, 3357, 0.4272), (4, 2, 3158, 4067, 0.3581), (2, 1, 1621, 2424, 0.3521), (5, 2, 3888, 4835, 0.3192)],
    "D-006": [(21, 8, 16074, 17060, 0.5331), (64, 22, 47810, 48731, 0.5004), (14, 5, 10845, 11545, 0.4772), (13, 5, 10086, 11018, 0.4696)],
    "D-007": [(69, 24, 51414, 52396, 0.7094), (27, 10, 20555, 21506, 0.6105), (85, 29, 63419, 64381, 0.5954), (84, 29, 62565, 63528, 0.5902)],
    "D-008": [(28, 10, 21315, 22262, 0.2598), (69, 24, 51414, 52396, 0.2535), (58, 20, 43256, 44253, 0.2478), (12, 5, 9290, 10243, 0.2041)],
    "D-009": [(85, 29, 63419, 64381, 0.3099), (27, 10, 20555, 21506, 0.2798), (69, 24, 51414, 52396, 0.2738), (84, 29, 62565, 63528, 0.2571)],
    "D-010": [(2, 1, 1621, 2424, 0.5196), (1, 1, 767, 1763, 0.434), (5, 2, 3888, 4835, 0.4061), (85, 29, 63419, 64381, 0.4043)],
}
D002_RANKS_5_TO_8 = [(84, 29, 62565, 63528, 0.3387), (85, 29, 63419, 64381, 0.3114), (70, 24, 52277, 53236, 0.2567), (35, 12, 26295, 26966, 0.2448)]
SEEDS = {
    "D-001": "According to the document, by how much have global temperatures risen since the late 19th century?",
    "D-002": "In what year was the Paris Agreement adopted?",
    "D-003": "According to the document, what is essential regarding countries' nationally determined contributions (NDCs) for meeting the Paris Agreement's goals?",
    "D-004": "What does DAC stand for, and what does it do?",
    "D-005": "How does the document contrast coal and natural gas in terms of carbon intensity?",
    "D-006": "What are the advantages of putting solar panels on lakes or reservoirs?",
    "D-007": "What is the supreme decision-making body of the convention that provides the framework for negotiating the Kyoto Protocol and the Paris Agreement?",
    "D-008": "What does the Kigali Amendment target?",
    "D-009": "In what year was the IPCC established?",
    "D-010": "What share of global greenhouse gas emissions comes from aviation?",
}

def as_retrieved(spec):
    """C4 records: rank, chunk_id, doc_id, page, score, text (the C4 -> C5 interface)."""
    return [{"rank": i, "chunk_id": f"climate:{SNAPSHOT_ID}:{PARAMS_HASH}:{seq:05d}", "doc_id": "climate",
             "page": p, "score": sc, "text": DOC_TEXT[a:b]} for i, (seq, p, a, b, sc) in enumerate(spec, 1)]

RETRIEVED4 = as_retrieved(TOP4["D-002"])                       # the 4 fixed chunks for D-002
RETRIEVED8 = as_retrieved(TOP4["D-002"] + D002_RANKS_5_TO_8)   # same question, k = 8
RETRIEVED_D009 = as_retrieved(TOP4["D-009"])                   # unanswerable question; two chunks share page 29
Q_D002, Q_D009 = SEEDS["D-002"], SEEDS["D-009"]

# ---- the template file prompts/answer_v1.txt (06 §3.3 with two placeholders)
ANSWER_V1_TXT = (
    "SYSTEM:\n"
    "You answer questions using ONLY the numbered sources below.\n"
    "Rules:\n"
    "1. Every factual sentence must end with one or more citations in the form [doc_id:page],\n"
    "   copied exactly from the source headers, e.g. [climate:10].\n"
    "2. If the sources do not contain the answer, reply exactly:\n"
    "   \"I cannot answer this from the provided documents.\"\n"
    "   Do not use outside knowledge.\n"
    "3. Do not cite a source that does not support the sentence.\n"
    "4. Be concise: at most 5 sentences.\n"
    "\n"
    "USER:\n"
    "Sources:\n"
    "{sources}\n"
    "\n"
    "Question: {question}\n"
    "Answer:\n")
ABSTAIN = "I cannot answer this from the provided documents."

def split_template(text):
    """'SYSTEM:\n<system>\nUSER:\n<user pattern>' -> (system, user_pattern). Given."""
    head, user = text.split("USER:\n", 1)
    assert head.startswith("SYSTEM:\n") and head.endswith("\n\n"), "template must start with SYSTEM: and have a blank line before USER:"
    return head[len("SYSTEM:\n"):-1], user.rstrip("\n")

def split_sentences(text):
    """Heuristic (given). A citation right after the end punctuation is moved before it; then split after . ! ? + space."""
    t = re.sub(r"([.!?])((?:\s*\[[^\[\]\n]*\])+)", r"\2\1", text.strip())
    return [s for s in re.split(r"(?<=[.!?])\s+", t) if s.strip()]

class FakeGroq:
    """Stands in for groq.Groq: records each call and returns a canned reply shaped like groq 1.7.0's
    ChatCompletion. No network. The default reply is a hand-written toy answer."""
    def __init__(self, content="The Paris Agreement was adopted in 2015 [climate:10].", finish_reason="stop",
                 prompt_tokens=1000, completion_tokens=60, reasoning_tokens=40):
        self.content, self.finish_reason = content, finish_reason
        self.prompt_tokens, self.completion_tokens, self.reasoning_tokens = prompt_tokens, completion_tokens, reasoning_tokens
        self.calls = []
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))
    def _create(self, **kwargs):
        self.calls.append(kwargs)
        details = None if self.reasoning_tokens is None else SimpleNamespace(reasoning_tokens=self.reasoning_tokens)
        return SimpleNamespace(
            choices=[SimpleNamespace(index=0, finish_reason=self.finish_reason,
                                     message=SimpleNamespace(role="assistant", content=self.content, reasoning=None))],
            usage=SimpleNamespace(prompt_tokens=self.prompt_tokens, completion_tokens=self.completion_tokens,
                                  completion_tokens_details=details))

print(len(pages), "pages | D-002 top-4 (rank, page, score):", [(r["rank"], r["page"], r["score"]) for r in RETRIEVED4])
print("D-009 top-4 pages:", [r["page"] for r in RETRIEVED_D009])

## Task 1: from C4's records to chat messages

Write two functions.
- `build_context(retrieved)` returns a tuple `(sources_text, chunk_ids)`. Sort the records by `rank` (do **not** trust the list order). Each block is a header line `[i] (doc_id:page)`, a newline, then the chunk text; `i` counts from 1 in rank order. Join the blocks with one empty line (`"\n\n"`). `chunk_ids` lists the chunk ids in the same order.
- `build_messages(question, retrieved, template_text=ANSWER_V1_TXT)` returns a list of two dicts: `{"role": "system", "content": system}` and `{"role": "user", "content": ...}`. Get `system` and the user pattern from `split_template(template_text)`. Fill the pattern's `{sources}` and `{question}`.

Then set:
- `tokens_k4 = n_tok(system content) + n_tok(user content)` for `build_messages(Q_D002, RETRIEVED4)`;
- `TEMPLATE_SHA256 = sha256_hex(ANSWER_V1_TXT)`.

**Predict first.** P7's lab prompt with four 1,000-character slices was 958 tokens. These four chunks are 657 to 982 characters long (the recursive splitter cuts at separators). Predict `tokens_k4`.

In [ ]:
my_prediction_tokens_k4 = None    # replace None with your number, BEFORE you build the messages

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c5_measure.py).

In [ ]:
# CHECK Task 1 (do not edit)
assert my_prediction_tokens_k4 is not None, "write your prediction first"
_src, _ids = build_context(RETRIEVED4)
assert isinstance(_src, str) and isinstance(_ids, list), "build_context returns (sources_text, chunk_ids)"
assert re.findall(r"^\[(\d+)\] \(([^:()]+):(\d+)\)$", _src, re.M) == [("1", "climate", "24"), ("2", "climate", "23"), ("3", "climate", "10"), ("4", "climate", "6")], \
    "headers must be '[i] (doc_id:page)' on their own line, numbered 1..4 in RANK order"
assert _ids == [r["chunk_id"] for r in RETRIEVED4], "chunk_ids must follow rank order"
_shuffled = [RETRIEVED4[2], RETRIEVED4[0], RETRIEVED4[3], RETRIEVED4[1]]
assert build_context(_shuffled) == (_src, _ids), "sort by the 'rank' field: a shuffled list must give the same context"
_m = build_messages(Q_D002, RETRIEVED4)
assert [x["role"] for x in _m] == ["system", "user"], "two messages: system, then user"
assert _m[0]["content"] == split_template(ANSWER_V1_TXT)[0], "system content comes from the template file"
assert _m[1]["content"].startswith("Sources:\n[1] (climate:24)\n") and _m[1]["content"].endswith("Question: " + Q_D002 + "\nAnswer:"), \
    "user content: 'Sources:', the blocks, a blank line, 'Question: ...', then 'Answer:'"
assert _fp(_m) == "820b2956c6ec6f35", "structure is right but a separator differs: one empty line between blocks"
assert _fp(tokens_k4) == "477f4b2cdd3fe2b9", f"tokens_k4 = {tokens_k4}: count n_tok(system) + n_tok(user)"
assert _fp(TEMPLATE_SHA256) == "1dd899a0d65f877a", "TEMPLATE_SHA256 = sha256_hex(ANSWER_V1_TXT)"
assert sha256_hex(ANSWER_V1_TXT.replace("5 sentences", "5  sentences")) != TEMPLATE_SHA256
print(f"Task 1 OK. tokens_k4 = {tokens_k4}; you predicted {my_prediction_tokens_k4}")
print(_m[1]["content"][:300])

**One sentence.** Why does the trace store `TEMPLATE_SHA256`, and not just the name `answer_v1`?

*Template:* "Because if someone ____ the file, the name stays the same but the hash ____."

*Your answer:*

## Task 2: one configurable model call, with an assert on content

Write `call_llm(client, messages, *, model_id, temperature=0.0, max_tokens=512, reasoning_effort="low")`.
- Call `client.chat.completions.create(...)` with exactly these keyword arguments: `model`, `messages`, `temperature`, `max_tokens`, `reasoning_effort`. The model id comes from the argument (in real runs from `.env`), never hardcoded.
- Time the call with `time.perf_counter()`.
- Read the text from `choices[0].message.content` (it may be `None`). **Assert** that it is not empty after `.strip()`; the assert message must include `finish_reason`.
- Return a dict with exactly these keys: `text`, `finish_reason`, `input_tokens` (`usage.prompt_tokens`), `output_tokens` (`usage.completion_tokens`), `reasoning_tokens` (`usage.completion_tokens_details.reasoning_tokens`, or `None` when `completion_tokens_details` is `None`), `latency_ms`, and `params` (a dict of `model`, `temperature`, `max_tokens`, `reasoning_effort`).

**Recall first.** In session #2 (O2), what came back from `gpt-oss-20b` at `max_tokens=12`, and why?

*Template:* "It returned ____, because the model spent ____ on ____."

*Your answer:*

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c5_measure.py).

In [ ]:
# CHECK Task 2 (do not edit). FakeGroq: no network, canned hand-written replies.
_fk = FakeGroq()
_r = call_llm(_fk, _m, model_id="fake-model-x")
_kw = _fk.calls[0]
assert set(_kw) == {"model", "messages", "temperature", "max_tokens", "reasoning_effort"}, f"keyword arguments sent: {sorted(_kw)}"
assert _kw["model"] == "fake-model-x", "use the model_id argument; never hardcode a model"
assert (_kw["temperature"], _kw["max_tokens"], _kw["reasoning_effort"]) == (0.0, 512, "low"), "defaults: 0.0, 512, 'low'"
assert _kw["messages"] is _m, "pass the messages through unchanged"
assert set(_r) == {"text", "finish_reason", "input_tokens", "output_tokens", "reasoning_tokens", "latency_ms", "params"}, f"keys: {sorted(_r)}"
assert (_r["text"], _r["finish_reason"], _r["input_tokens"], _r["output_tokens"], _r["reasoning_tokens"]) == \
    ("The Paris Agreement was adopted in 2015 [climate:10].", "stop", 1000, 60, 40)
assert _r["params"] == {"model": "fake-model-x", "temperature": 0.0, "max_tokens": 512, "reasoning_effort": "low"}
assert isinstance(_r["latency_ms"], float) and _r["latency_ms"] >= 0
assert call_llm(FakeGroq(reasoning_tokens=None), _m, model_id="m")["reasoning_tokens"] is None, \
    "completion_tokens_details can be None: return reasoning_tokens = None, do not crash"
_fk2 = FakeGroq()
call_llm(_fk2, _m, model_id="m", max_tokens=256, temperature=0.3, reasoning_effort="medium")
assert (_fk2.calls[0]["max_tokens"], _fk2.calls[0]["temperature"], _fk2.calls[0]["reasoning_effort"]) == (256, 0.3, "medium"), "pass the arguments through"
for _bad in ({"content": ""}, {"content": None}, {"content": "  \n"}, {"content": "", "finish_reason": "length"}):
    try:
        call_llm(FakeGroq(**_bad), _m, model_id="m")
        raise RuntimeError(f"no AssertionError for reply {_bad}: an empty answer must never pass silently")
    except AssertionError as e:
        assert str(_bad.get("finish_reason", "stop")) in str(e), "the assert message must include finish_reason"
print("Task 2 OK")

## Task 3: parse the citations out of an answer

Write `parse_citations(text)` that returns a tuple `(citations, malformed)`.
- `citations`: one dict per **distinct** `[doc_id:page]` tag, in order of first appearance: `{"marker": "[climate:10]", "doc_id": "climate", "page": 10}`. `page` is an `int`. A tag is valid only in the exact form `[` + letters/digits/`_`/`-` + `:` + digits + `]`, with no spaces.
- `malformed`: every other bracket tag `[...]`, and every `(doc:page)` look-alike in round brackets, in order of appearance, as the exact text found. Examples: `[4]`, `[climate: 10]`, `(climate:10)`, `[climate:10, climate:24]`.

Hint for the pattern search: `re.finditer` and `re.fullmatch`.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c5_measure.py).

In [ ]:
# CHECK Task 3 (do not edit). Hand-written toy answers, not model outputs.
assert parse_citations("The Paris Agreement was adopted in 2015 [climate:10].") == \
    ([{"marker": "[climate:10]", "doc_id": "climate", "page": 10}], []), "one valid tag; page is an int"
assert parse_citations("Adopted in 2015 [climate:10][climate:24]. It was in 2015 [climate:10].")[0] == \
    [{"marker": "[climate:10]", "doc_id": "climate", "page": 10}, {"marker": "[climate:24]", "doc_id": "climate", "page": 24}], \
    "distinct tags in order of FIRST appearance (the repeated [climate:10] is listed once)"
assert parse_citations("Adopted in 2015 [Climate:10].")[0][0]["doc_id"] == "Climate", "keep the doc_id exactly as written"
assert parse_citations("Adopted in 2015.") == ([], [])
assert parse_citations("Adopted in 2015 [4].") == ([], ["[4]"]), "a block number is not a citation: report it as malformed"
assert parse_citations("Adopted in 2015 [climate: 10].") == ([], ["[climate: 10]"]), "a space inside the tag makes it malformed"
assert parse_citations("Adopted in 2015 (climate:10).") == ([], ["(climate:10)"]), "a copied header '(doc:page)' is malformed"
assert parse_citations("Adopted in 2015 [climate:p10].") == ([], ["[climate:p10]"])
assert parse_citations("A [climate:10] B [2] C (climate:6) D [x:1]")[1] == ["[2]", "(climate:6)"], "malformed in order of appearance"
assert _fp(parse_citations("The Paris Agreement was adopted in 2015 [climate:10, climate:24].")) == "48e8934be17570b4", \
    "two citations inside ONE bracket are not the [doc_id:page] form"
print("Task 3 OK")

## Task 4: are the cited IDs a subset of the retrieved IDs?

A citation names a **page**; the retriever returned **chunks**. Write `check_citations(citations, retrieved)` that returns a dict with exactly three keys:
- `"resolved"`: a copy of each citation dict with one more key, `"resolved_chunk_ids"`: the ids of the retrieved chunks whose `(doc_id, page)` equals the citation's, in **rank** order (empty list if none);
- `"invalid"`: the `marker` of every citation that resolves to no chunk (**cited but not retrieved**), in citation order;
- `"unused_chunk_ids"`: the retrieved chunk ids, in rank order, that no citation resolves to (**retrieved but not used**).

Never drop an invalid citation silently: report it.

**Predict first.** For D-009, the hand-written answer `"The IPCC was established in 1988 [climate:29]."` is checked against `RETRIEVED_D009`. How many chunk ids does `[climate:29]` resolve to?

In [ ]:
my_prediction_resolved_29 = None    # a number, BEFORE you write the function

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c5_measure.py).

In [ ]:
# CHECK Task 4 (do not edit). Hand-written toy answers, not model outputs.
assert my_prediction_resolved_29 is not None, "write your prediction first"
_c = check_citations(parse_citations("The Paris Agreement was adopted in 2015 [climate:10].")[0], RETRIEVED4)
assert set(_c) == {"resolved", "invalid", "unused_chunk_ids"}
assert _c["resolved"][0]["resolved_chunk_ids"] == [RETRIEVED4[2]["chunk_id"]], "page 10 is rank 3"
assert _c["invalid"] == [] and _c["unused_chunk_ids"] == [RETRIEVED4[i]["chunk_id"] for i in (0, 1, 3)]
_c2 = check_citations(parse_citations("Adopted in 2015 [climate:12].")[0], RETRIEVED4)
assert _c2["invalid"] == ["[climate:12]"], "page 12 was never retrieved: report it (do not drop it silently)"
assert len(_c2["resolved"]) == 1 and _c2["resolved"][0]["resolved_chunk_ids"] == [], "keep the invalid citation in 'resolved' with an empty list"
assert check_citations(parse_citations("x [Climate:10].")[0], RETRIEVED4)["invalid"] == ["[Climate:10]"], "doc ids are case-sensitive"
assert check_citations([], RETRIEVED4)["unused_chunk_ids"] == [r["chunk_id"] for r in RETRIEVED4]
_c3 = check_citations(parse_citations("x [climate:10].")[0], list(reversed(RETRIEVED4)))
assert _c3["unused_chunk_ids"] == [RETRIEVED4[i]["chunk_id"] for i in (0, 1, 3)], "unused ids in RANK order, whatever the list order"
_c4 = check_citations(parse_citations("The IPCC was established in 1988 [climate:29].")[0], RETRIEVED_D009)
assert _fp(_c4) == "d198d0a723892dc6", "a page citation resolves to EVERY retrieved chunk on that page, in rank order"
print("Task 4 OK. [climate:29] resolves to", len(_c4["resolved"][0]["resolved_chunk_ids"]), "chunk ids; you predicted", my_prediction_resolved_29)
print("Is the 1988 answer supported by the sources? The subset check says:", _c4["invalid"] == [])

**One sentence.** The subset check passed for the 1988 answer. Does that mean the answer is supported by the sources?

*Template:* "No / Yes, because the check only tests ____, not ____."

*Your answer:*

## Task 5: the answer part of the record

Write two functions.
- `is_abstention(text)`: `True` when the text is the fixed `ABSTAIN` sentence, ignoring case, surrounding spaces and quotes, runs of whitespace, and a final full stop. Anything else is `False` (you wrote this in P7; write it again from memory).
- `make_answer(text, retrieved, finish_reason="stop")` returns a dict with exactly these keys. `finish_reason` is the provider's value for this reply (Task 6 passes it in); a hand-written toy answer keeps the default `"stop"`.
  - `text`; `abstained` (`is_abstention(text)`);
  - `citations` (the `resolved` list from Task 4), `invalid_citations`, `malformed_markers`, `unused_chunk_ids`;
  - `uncited_sentences`: the number of sentences from `split_sentences(text)` that contain no valid tag (0 when abstained);
  - `citation_subset_ok`: `True` when there are no invalid citations **and** no malformed tags;
  - `flags`: a **sorted** list built from these rules:
    - `"empty"`: the text is empty after `.strip()`;
    - `"cited_not_retrieved"`: at least one invalid citation;
    - `"malformed_tag"`: at least one malformed tag;
    - `"no_citation"`: non-empty, not an abstention, and no valid citation at all;
    - `"uncited_sentence"`: at least one valid citation **and** at least one uncited sentence;
    - `"abstention_inside_answer"`: not an abstention, but the normalised abstention sentence appears inside the normalised text;
    - `"length_stop"`: `finish_reason == "length"` (the reply hit the `max_tokens` cap, so it may be cut off);
    - `"unexpected_finish_reason"`: `finish_reason` is anything other than `"stop"` or `"length"`, including `None`.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c5_measure.py).

In [ ]:
# CHECK Task 5 (do not edit). Every answer below is a HAND-WRITTEN TOY ANSWER, not a model output.
TOY = [
    ("T01 good", "The Paris Agreement was adopted in 2015 [climate:10].", RETRIEVED4),
    ("T02 fabricated page", "The Paris Agreement was adopted in 2015 [climate:12].", RETRIEVED4),
    ("T03 wrong doc id", "The Paris Agreement was adopted in 2015 [paris:10].", RETRIEVED4),
    ("T04 case differs", "The Paris Agreement was adopted in 2015 [Climate:10].", RETRIEVED4),
    ("T05 no citation", "The Paris Agreement was adopted in 2015.", RETRIEVED4),
    ("T06 one of two uncited", "The Paris Agreement was adopted in 2015 [climate:10]. It aims to limit warming to well below 2 degrees Celsius.", RETRIEVED4),
    ("T07 block number", "The Paris Agreement was adopted in 2015 [4].", RETRIEVED4),
    ("T08 space inside", "The Paris Agreement was adopted in 2015 [climate: 10].", RETRIEVED4),
    ("T09 header copy ()", "The Paris Agreement was adopted in 2015 (climate:10).", RETRIEVED4),
    ("T10 two in one []", "The Paris Agreement was adopted in 2015 [climate:10, climate:24].", RETRIEVED4),
    ("T11 after full stop", "The Paris Agreement was adopted in 2015. [climate:10]", RETRIEVED4),
    ("T12 two valid tags", "The Paris Agreement was adopted in 2015 [climate:10][climate:24].", RETRIEVED4),
    ("T13 exact abstention", "I cannot answer this from the provided documents.", RETRIEVED4),
    ("T14 abstention variant", '  "i cannot answer this from the provided documents"  ', RETRIEVED4),
    ("T15 paraphrased refusal", "The sources do not say when the Paris Agreement was adopted.", RETRIEVED4),
    ("T16 abstention + tag", "I cannot answer this from the provided documents [climate:23].", RETRIEVED4),
    ("T17 empty", "", RETRIEVED4),
    ("T18 D-009 abstention", "I cannot answer this from the provided documents.", RETRIEVED_D009),
    ("T19 D-009 memory+valid tag", "The IPCC was established in 1988 [climate:29].", RETRIEVED_D009),
]
_A = {tid: make_answer(t, r) for tid, t, r in TOY}
_keys = {"text", "abstained", "citations", "invalid_citations", "malformed_markers", "unused_chunk_ids",
         "uncited_sentences", "citation_subset_ok", "flags"}
assert all(set(a) == _keys for a in _A.values()), f"keys must be exactly {sorted(_keys)}"
assert is_abstention(ABSTAIN) and is_abstention('  "I cannot   answer this from the provided documents"  '),     "is_abstention: ignore case, surrounding quotes and spaces, runs of whitespace, and a final full stop"
assert not is_abstention("I can't answer that from these sources."), "only the fixed sentence counts"
assert _A["T01 good"]["flags"] == [] and _A["T01 good"]["citation_subset_ok"] is True
assert _A["T02 fabricated page"]["flags"] == ["cited_not_retrieved"] and _A["T02 fabricated page"]["citation_subset_ok"] is False
assert _A["T05 no citation"]["flags"] == ["no_citation"], "a fact with no citation must be flagged"
assert _A["T05 no citation"]["citation_subset_ok"] is True, "no citation is NOT a subset failure (the empty set is a subset)"
assert _A["T06 one of two uncited"]["flags"] == ["uncited_sentence"] and _A["T06 one of two uncited"]["uncited_sentences"] == 1
assert _A["T07 block number"]["flags"] == ["malformed_tag", "no_citation"], "flags are sorted"
assert _A["T11 after full stop"]["uncited_sentences"] == 0, "use split_sentences: a tag after the full stop belongs to that sentence"
assert _A["T13 exact abstention"]["abstained"] and _A["T14 abstention variant"]["abstained"]
assert not _A["T15 paraphrased refusal"]["abstained"] and _A["T15 paraphrased refusal"]["flags"] == ["no_citation"]
assert _A["T16 abstention + tag"]["flags"] == ["abstention_inside_answer"]
assert _A["T17 empty"]["flags"] == ["empty"]
assert _fp([_A[t] for t, _, _ in TOY]) == "3133756f8e11d25f", \
    "one field differs from the reference for at least one toy answer; compare each rule with the task text"
for tid, a in _A.items():
    print(f"{tid:28s} subset_ok={a['citation_subset_ok']!s:5s} abstained={a['abstained']!s:5s} flags={a['flags']}")
print("Task 5 OK")

## Task 6: the record C6 will store, and the sanity script (02 evidence)

Write `make_record(qid, question, retrieved, messages, llm_result, template_text=ANSWER_V1_TXT)` that returns a dict with exactly these keys (the C5 part of the 06 §2.5 trace):
- `qid`, `question`;
- `context`: `{"chunk_ids": ..., "context_tokens": n_tok(sources_text), "tokenizer": "o200k_harmony", "truncated_chunks": 0}` (use `build_context`);
- `prompt`: `{"template_version": "answer_v1", "template_sha256": ..., "sha256": sha256_hex(json.dumps(messages, sort_keys=True, ensure_ascii=False)), "local_tokens": ..., "input_tokens": llm_result["input_tokens"]}`; `local_tokens` is the sum of `n_tok` over the message contents;
- `llm`: `{"provider": "groq", "model_id", "temperature", "max_tokens", "reasoning_effort"` (all four from `llm_result["params"]`; `model_id` is `params["model"]`), `"finish_reason", "cache_hit": False}`;
- `answer`: `make_answer(llm_result["text"], retrieved, finish_reason=llm_result["finish_reason"])` plus `output_tokens` and `reasoning_tokens` from `llm_result`;
- `latency_ms`: `{"llm": llm_result["latency_ms"]}`; `errors`: an empty list.

Then write `failing_qids(records)`: the `qid` of every record whose `answer.citation_subset_ok` is `False` **or** whose `errors` list is not empty, in input order. This is the "cited IDs ⊆ retrieved IDs" sanity script from the 02 card.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c5_measure.py).

In [ ]:
# CHECK Task 6 (do not edit). FakeGroq reply = hand-written toy answer T01.
_rec = make_record("D-002", Q_D002, RETRIEVED4, _m, call_llm(FakeGroq(), _m, model_id="openai/gpt-oss-20b"))
assert set(_rec) == {"qid", "question", "context", "prompt", "llm", "answer", "latency_ms", "errors"}, f"top-level keys: {sorted(_rec)}"
assert _rec["context"] == {"chunk_ids": [r["chunk_id"] for r in RETRIEVED4], "context_tokens": n_tok(build_context(RETRIEVED4)[0]),
                           "tokenizer": "o200k_harmony", "truncated_chunks": 0}
assert _rec["prompt"]["template_version"] == "answer_v1" and _rec["prompt"]["template_sha256"] == TEMPLATE_SHA256
assert _rec["prompt"]["sha256"] == sha256_hex(json.dumps(_m, sort_keys=True, ensure_ascii=False)), "prompt.sha256: hash the messages as the task says"
assert _rec["prompt"]["local_tokens"] == tokens_k4 and _rec["prompt"]["input_tokens"] == 1000
assert _rec["llm"] == {"provider": "groq", "model_id": "openai/gpt-oss-20b", "temperature": 0.0, "max_tokens": 512,
                       "reasoning_effort": "low", "finish_reason": "stop", "cache_hit": False},     f"llm part differs: {_rec['llm']} (model_id comes from params['model'])"
assert _rec["answer"] == {**make_answer("The Paris Agreement was adopted in 2015 [climate:10].", RETRIEVED4), "output_tokens": 60, "reasoning_tokens": 40}
assert list(_rec["latency_ms"]) == ["llm"] and _rec["errors"] == []
json.dumps(_rec)  # must be JSON-serialisable (it goes into results.jsonl)
def _toy(qid, text, err=None):
    r = make_record(qid, "q", RETRIEVED4, _m, {"text": text, "finish_reason": "stop", "input_tokens": None, "output_tokens": None,
        "reasoning_tokens": None, "latency_ms": 0.0, "params": {"model": "m", "temperature": 0.0, "max_tokens": 512, "reasoning_effort": "low"}})
    if err: r["errors"].append(err)
    return r
_recs = [_toy("T01", TOY[0][1]), _toy("T02", TOY[1][1]), _toy("T05", TOY[4][1], "empty answer (finish_reason=length)"),
         _toy("T07", TOY[6][1]), _toy("T13", TOY[12][1]), _toy("T12", TOY[11][1])]
assert failing_qids(_recs) == ["T02", "T05", "T07"], f"failing_qids gave {failing_qids(_recs)}"
assert failing_qids([]) == []
# A cut-off reply (hand-written toy): NOT empty, so call_llm passes, but the model stopped at the max_tokens cap.
_cut = make_record("D-002", Q_D002, RETRIEVED4, _m, call_llm(FakeGroq(
    content="The Paris Agreement was adopted in 2015 [climate:10]. It aims to limit warming to", finish_reason="length"), _m, model_id="m"))
assert "length_stop" in _cut["answer"]["flags"], \
    "finish_reason 'length' means the reply hit the max_tokens cap: flag it 'length_stop' (make_record passes llm_result['finish_reason'] to make_answer)"
assert "unexpected_finish_reason" not in _cut["answer"]["flags"], "'length' is an expected finish_reason: flag only 'length_stop'"
print("Task 6 OK")
print(json.dumps(_rec["answer"], indent=1)[:600])

## Task 7: retrieval or generation? (CR-11)

For each scenario, write two labels.
- `my_kind[x]`: `"retrieval"` (the evidence was not in the top-k), `"generation"` (the evidence was there, or nothing could be there, and the answer or its citations are wrong), `"both"`, or `"none"` (nothing is wrong).
- `my_caught_by[x]`: which v0 check would catch it: `"subset_check"`, `"no_citation_flag"`, `"retrieval_metric"` (needs the gold page), `"not_caught_in_v0"`, or `"nothing_to_catch"`.

| | Scenario (answers are hand-written toy answers) |
|---|---|
| a | D-002, gold page 10 retrieved at rank 3. The answer cites `[climate:12]`. |
| b | D-002, gold page 10 retrieved. The answer cites only `[climate:10]`; pages 24, 23 and 6 are unused. |
| c | D-009 (unanswerable). The answer is "The IPCC was established in 1988 [climate:29]." Page 29 was retrieved. |
| d | The gold page is not in the top-4. The answer is the exact abstention sentence. |
| e | D-002, gold page 10 retrieved. The answer is "The Paris Agreement was adopted in 2016 [climate:23]." |
| f | The gold page is not in the top-4. The answer is correct but has no citation. |

In [ ]:
my_kind = {"a": None, "b": None, "c": None, "d": None, "e": None, "f": None}
my_caught_by = {"a": None, "b": None, "c": None, "d": None, "e": None, "f": None}

In [ ]:
# CHECK Task 7 (do not edit)
_K = {"retrieval", "generation", "both", "none"}
_C = {"subset_check", "no_citation_flag", "retrieval_metric", "not_caught_in_v0", "nothing_to_catch"}
assert all(v in _K for v in my_kind.values()), f"each kind must be one of {sorted(_K)}"
assert all(v in _C for v in my_caught_by.values()), f"each caught_by must be one of {sorted(_C)}"
assert _fp(my_kind) == "694ce518d388e59b", "at least one kind is wrong. Ask: was the evidence in the top-4? Then: is the answer wrong?"
assert _fp(my_caught_by) == "f59019086a14da25", "at least one 'caught_by' is wrong. Run make_answer on the toy answer and look at the flags."
print("Task 7 OK")

**One sentence (scenario e).** The subset check passes for e. What kind of check would catch it, and which version owns that check?

*Template:* "A ____ check, which compares ____ with ____; it belongs to v____."

*Your answer:*

## Task 8: what if k doubles? (CR-08)

`RETRIEVED8` holds ranks 1–8 for D-002. Measure two things:
- `tokens_k8`: system + user tokens of `build_messages(Q_D002, RETRIEVED8)`;
- `unused_k8`: the number of `unused_chunk_ids` when the hand-written toy answer T01 (`"The Paris Agreement was adopted in 2015 [climate:10]."`) is checked against `RETRIEVED8`.

**Predict first:** both numbers.

In [ ]:
my_prediction_tokens_k8 = None    # BEFORE measuring
my_prediction_unused_k8 = None    # BEFORE measuring

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c5_measure.py).

In [ ]:
# CHECK Task 8 (do not edit)
assert my_prediction_tokens_k8 is not None and my_prediction_unused_k8 is not None, "write both predictions first"
assert _fp(tokens_k8) == "8fd238119777a31e", f"tokens_k8 = {tokens_k8} is not right yet"
assert _fp(unused_k8) == "7902699be42c8a8e", f"unused_k8 = {unused_k8} is not right yet"
print(f"k 4 -> 8: {tokens_k4} -> {tokens_k8} tokens (x{tokens_k8 / tokens_k4:.2f}); you predicted {my_prediction_tokens_k8}")
print(f"unused chunks for T01: 3 of 4 -> {unused_k8} of 8; you predicted {my_prediction_unused_k8}")
print("Task 8 OK")

**Two sentences.** (1) Why is the token ratio below 2? (2) Does a high `unused` count mean the answer is bad?

*Template:* "(1) Below 2 because ____ stay fixed. (2) No / Yes, because ____."

*Your answer:*

**What if `max_tokens` halves to 256?** Do not run it here. Write your prediction for the answer and for its citations.

*Template:* "The answer might ____, because the reasoning tokens ____. The citation check might then flag ____."

*Your answer:*

## Task 9 (optional, calls Groq): the sanity check on 10 queries (02 evidence)

**Not run by the course author. Every output of this cell is UNVERIFIED until you run it.** It sends the 10 draft seed questions (06 §6.7), each with its 4 fixed retrieved chunks from the setup cell, through your `build_messages` → `call_llm` → `make_record`, writes one JSON record per line to `C5_groq_run_<date>.jsonl` next to this notebook, and prints `failing_qids`.

Cost, measured locally (lesson S4/S8): about 8,430 input tokens in total before the provider's template overhead, plus at most 10 × 512 output tokens. Look up today's Groq price before running.

Set `RUN_GROQ = True` to run it. It reads `.env` and never prints the key. **Before running**, write your prediction:

*My prediction:* abstentions on ____ ; failing qids: ____ .

In [ ]:
RUN_GROQ = False   # optional; set True to spend Groq tokens (see the cost line above)

if RUN_GROQ:
    import os, datetime
    from dotenv import load_dotenv, find_dotenv
    from groq import Groq
    load_dotenv(find_dotenv(usecwd=True))              # rag_learning/.env; the key is never printed
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
    MODEL_ID = os.environ["GROQ_MODEL_ID"]              # never hardcode the model id
    run = []
    for qid, question in SEEDS.items():
        retrieved = as_retrieved(TOP4[qid])
        messages = build_messages(question, retrieved)
        try:
            rec = make_record(qid, question, retrieved, messages, call_llm(client, messages, model_id=MODEL_ID))
        except AssertionError as e:                    # an empty answer is a failure, not a skip
            rec = make_record(qid, question, retrieved, messages, {
                "text": "", "finish_reason": None, "input_tokens": None, "output_tokens": None, "reasoning_tokens": None,
                "latency_ms": None, "params": {"model": MODEL_ID, "temperature": 0.0, "max_tokens": 512, "reasoning_effort": "low"}})
            rec["errors"].append(str(e))
        run.append(rec)
        a = rec["answer"]
        print(f"{qid} abstained={a['abstained']} cites={[c['marker'] for c in a['citations']]} flags={a['flags']} "
              f"finish={rec['llm']['finish_reason']} out={a['output_tokens']} errors={rec['errors']}")
    out = f"C5_groq_run_{datetime.date.today()}.jsonl"
    with open(out, "w", encoding="utf-8") as f:
        for r in run:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("saved", out, "| failing_qids:", failing_qids(run), "| PASS" if not failing_qids(run) else "| FAIL")
else:
    print("Task 9 skipped (RUN_GROQ = False).")

## Log it

Copy into your session log: your predictions next to the measured `tokens_k4`, `resolved` count, `tokens_k8` and `unused_k8`; your sentences for Tasks 1, 4, 7 and 8. If you ran Task 9, paste the printed lines and label them **one run, n = 10 draft seeds, not an evaluation result**. Read every answer yourself: the subset check cannot tell you whether a cited page really supports the sentence.